In [1]:
5+5

10

In [11]:
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from loguru import logger
from scipy.signal import resample_poly


# =========================================================
# CONFIG
# =========================================================

BASE_DIR = Path("..").resolve()

INPUT_METADATA_PATH = (
    BASE_DIR / "data/final_dataset_metadata.jsonl"
)

OUTPUT_DATASET_DIR = (
    BASE_DIR / "data" / "ready_dataset"
)

OUTPUT_WAV_DIR = (
    OUTPUT_DATASET_DIR / "wavs"
)

METADATA_CSV_PATH = (
    OUTPUT_DATASET_DIR / "metadata.csv"
)

In [12]:



# =========================================================
# AUDIO UTILS
# =========================================================

def normalize_audio(
    audio: np.ndarray
):

    peak = np.max(np.abs(audio))

    if peak > 0:

        audio = audio / peak

    return audio


def convert_to_mono(
    audio: np.ndarray
):

    if audio.ndim > 1:

        audio = audio.mean(axis=1)

    return audio


def resample_audio(
    audio: np.ndarray,
    original_sr: int,
    target_sr: int,
):

    if original_sr == target_sr:

        return audio

    gcd = np.gcd(original_sr, target_sr)

    audio = resample_poly(
        audio,
        target_sr // gcd,
        original_sr // gcd
    )

    return audio


# =========================================================
# LOAD JSONL
# =========================================================

def load_jsonl(
    jsonl_path: Path
):

    records = []

    with open(
        jsonl_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            try:

                records.append(
                    json.loads(line)
                )

            except Exception as e:

                logger.warning(
                    f"Failed parsing line: {e}"
                )

    return records

# FORMATTER SERVICE

In [13]:

class DatasetFormatterService:

    def __init__(

        self,

        metadata_path: Path,

        output_dataset_dir: Path,

        target_sample_rate: int = 16000,

        mono: bool = True,

        normalize: bool = True,
    ):

        self.metadata_path = metadata_path

        self.output_dataset_dir = output_dataset_dir

        self.output_wav_dir = (
            output_dataset_dir / "wavs"
        )

        self.target_sample_rate = (
            target_sample_rate
        )

        self.mono = mono

        self.normalize = normalize

        # create dirs
        self.output_wav_dir.mkdir(
            parents=True,
            exist_ok=True
        )

    # =====================================================
    # PROCESS SINGLE AUDIO
    # =====================================================

    def process_audio(

        self,

        input_audio_path: Path,

        output_audio_path: Path,
    ):

        audio, sr = sf.read(input_audio_path)

        audio = audio.astype(np.float32)

        # =============================================
        # MONO
        # =============================================

        if self.mono:

            audio = convert_to_mono(audio)

        # =============================================
        # RESAMPLE
        # =============================================

        audio = resample_audio(
            audio,
            sr,
            self.target_sample_rate
        )

        # =============================================
        # NORMALIZE
        # =============================================

        if self.normalize:

            audio = normalize_audio(audio)

        # =============================================
        # SAVE WAV
        # =============================================

        sf.write(
            output_audio_path,
            audio,
            self.target_sample_rate,
            subtype="PCM_16"
        )

    # =====================================================
    # FORMAT DATASET
    # =====================================================

    def build_dataset(self):

        logger.info(
            "Loading metadata..."
        )

        records = load_jsonl(
            self.metadata_path
        )

        logger.info(
            f"Loaded samples: {len(records)}"
        )

        csv_rows = []

        processed_count = 0
        skipped_count = 0

        for idx, record in enumerate(records):

            try:

                # =====================================
                # GET AUDIO
                # =====================================

                original_audio_path = (
                    BASE_DIR /
                    record["audio_path"]
                ).resolve()

                if not original_audio_path.exists():

                    logger.warning(
                        f"Audio missing: "
                        f"{original_audio_path}"
                    )

                    skipped_count += 1
                    continue

                # =====================================
                # TRANSCRIPTION
                # =====================================

                transcription = (
                    record["text"]
                    .strip()
                )

                if len(transcription) == 0:

                    logger.warning(
                        f"Empty transcription | "
                        f"id={record['audio_id']}"
                    )

                    skipped_count += 1
                    continue

                # =====================================
                # OUTPUT NAME
                # =====================================

                filename = (
                    f"sample_{idx:06d}.wav"
                )

                output_audio_path = (
                    self.output_wav_dir /
                    filename
                )

                # =====================================
                # PROCESS AUDIO
                # =====================================

                self.process_audio(
                    original_audio_path,
                    output_audio_path
                )

                # =====================================
                # CSV ROW
                # =====================================

                csv_rows.append({

                    "audio": (
                        f"wavs/{filename}"
                    ),

                    "transcription": (
                        transcription
                    ),
                })

                processed_count += 1

                logger.success(
                    f"Processed | "
                    f"{filename}"
                )

            except Exception as e:

                logger.exception(
                    f"Failed processing sample | "
                    f"id={record.get('audio_id')}"
                )

                skipped_count += 1

        # =================================================
        # SAVE CSV
        # =================================================

        df = pd.DataFrame(csv_rows)

        df.to_csv(
            METADATA_CSV_PATH,
            index=False,
            encoding="utf-8-sig"
        )

        logger.success(
            f"Dataset formatting completed | "
            f"processed={processed_count} | "
            f"skipped={skipped_count}"
        )

        logger.info(
            f"Metadata saved to: "
            f"{METADATA_CSV_PATH}"
        )

        logger.info(
            f"Wavs saved to: "
            f"{self.output_wav_dir}"
        )





In [14]:
formatter = DatasetFormatterService(

    metadata_path=INPUT_METADATA_PATH,

    output_dataset_dir=OUTPUT_DATASET_DIR,

    target_sample_rate=16000,

    mono=True,

    normalize=True,
)

formatter.build_dataset()

2026-05-12 11:21:43.313 | INFO     | __main__:build_dataset:100 - Loading metadata...
2026-05-12 11:21:43.316 | INFO     | __main__:build_dataset:108 - Loaded samples: 3
2026-05-12 11:21:43.337 | SUCCESS  | __main__:build_dataset:198 - Processed | sample_000000.wav
2026-05-12 11:21:43.350 | SUCCESS  | __main__:build_dataset:198 - Processed | sample_000001.wav
2026-05-12 11:21:43.364 | SUCCESS  | __main__:build_dataset:198 - Processed | sample_000002.wav
2026-05-12 11:21:43.368 | SUCCESS  | __main__:build_dataset:224 - Dataset formatting completed | processed=3 | skipped=0
2026-05-12 11:21:43.369 | INFO     | __main__:build_dataset:230 - Metadata saved to: D:\GAN_AI\Synthetic-Speech-Data-Pipeline-For-STT\data\ready_dataset\metadata.csv
2026-05-12 11:21:43.370 | INFO     | __main__:build_dataset:235 - Wavs saved to: D:\GAN_AI\Synthetic-Speech-Data-Pipeline-For-STT\data\ready_dataset\wavs
